# What is Unsupervised Learning?

**Unsupervised learning** is a class of machine learning methods that does not rely on labeled outcome variables. Instead of predicting a known target (e.g., fake vs. true), these models aim to **discover patterns, structure, or relationships within the data itself**.

Common types of unsupervised learning methods include:
- **Clustering models** (e.g., K-Means, GMM, HDBSCAN)
- **Dimensionality reduction** (e.g., PCA, SVD)
- **Topic modeling** (e.g., LDA)
- **Representation learning** (e.g., embeddings, neural networks)

In contrast to supervised learning, which assumes a known relationship between predictors and an outcome variable, unsupervised learning does not make assumptions about labels. Instead, it explores how data points are organized in feature space.

A key advantage of unsupervised learning is that it allows us to:
- uncover hidden structure in the data
- identify natural groupings or clusters
- analyze similarities and differences between observations

Unlike supervised models, the outputs are not direct predictions (e.g., 0 or 1), but rather:
- cluster assignments
- similarity relationships
- or probabilistic memberships (e.g., in Gaussian Mixture Models)

In this project, unsupervised learning is used to investigate whether fake and true news articles form distinct groups or exhibit overlapping semantic structure. This is particularly important in fake news detection (FND), where high predictive accuracy in supervised models may not necessarily reflect meaningful separation in the underlying data structure.

## K-Means Clustering

K-Means is one of the most widely used unsupervised learning algorithms for clustering.
The goal of K-Means is to partition the dataset into **K distinct clusters**, where each data point belongs to the cluster with the nearest mean (centroid).

### How K-Means Works

The algorithm follows an iterative process:

1. **Initialize K centroids** randomly in the feature space
2. **Assign each data point** to the nearest centroid (based on distance, typically Euclidean distance)
3. **Update centroids** by computing the mean of all points assigned to each cluster
4. Repeat steps 2–3 until the centroids stabilize (convergence)

### Objective Function

K-Means aims to minimize the **within-cluster variance**, also known as inertia:

- Points within the same cluster should be as similar as possible
- Clusters should be compact and well-separated

### Key Characteristics

- Produces **hard cluster assignments** (each point belongs to one cluster)
- Requires specifying the number of clusters **K** in advance
- Sensitive to:
  - initialization of centroids
  - scaling of features
  - high-dimensional data

### Limitations

- Assumes clusters are **spherical and evenly sized**
- Struggles with:
  - overlapping clusters
  - non-linear structures
- May not perform well when the true structure of the data is complex

### Application in This Project

In this project, K-Means is applied to text representations (TF-IDF and GPT embeddings) to explore whether fake and true news articles form distinct clusters in semantic space.

By analyzing cluster composition (e.g., proportion of fake vs. true articles within each cluster), we can evaluate whether fake news naturally separates from real news or exhibits overlapping structure.


### Feature Representation

In this section, clustering is performed using **TF-IDF-based representations** of the text data.

The high-dimensional TF-IDF matrix is reduced using **Truncated SVD**, producing a dense representation (`X_reduced`) that captures dominant textual patterns.

This approach emphasizes **word frequency and writing style**, rather than semantic meaning, which allows us to explore whether surface-level features alone can reveal meaningful structure in fake news data.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import umap
import matplotlib.pyplot as plt

# Evaluate different values of K for K-means
for k in [5, 10, 15, 20]:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10)  # run kmeans 10 times, each with different random initial cluster centers. keep the lowest total within-cluster variance
    labels = kmeans.fit_predict(X_reduced)
    score = silhouette_score(X_reduced,
                             labels)  # how well each data point fits within its assigned cluster compared to other clusters?
    print(f"k = {k}, silhouette_score = {score:.4f}")

# k = 15 gives the highest silhouette score
# The score is still low, suggesting overlapping semantic structure
km = KMeans(
    n_clusters=15,
    random_state=42,
    n_init=10
)
labels = km.fit_predict(X_reduced)
df["cluster"] = labels

# check cluster sizes
print(df["cluster"].value_counts())

# PCA visualization
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_reduced)

plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab20', s=5)
plt.title("K-means Clusters (PCA projection)")
plt.show()

# UMAP visualization
# UMAP reduces high-dimensional data into 2D while preserving local structure
# This helps reveal clusters and non-linear relationships in the data
reducer = umap.UMAP(random_state=42)
X_2d = reducer.fit_transform(X_reduced)

plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab20', s=5)
plt.title("K-means Clusters (UMAP)")
plt.show()

# The silhouette scores are relatively low, indicating that the news articles do not form perfectly separated clusters. This suggests that fake and true news may overlap in semantic space, with themes and writing styles shared across clusters. The PCA and UMAP plots provide a lower-dimensional view of this structure, but they should be interpreted cautiously because dimensionality reduction can distort distances.

## Clustering with GPT Embeddings

### Feature Representation (GPT Embeddings)

In this section, clustering is performed using **GPT-based text embeddings**.

Unlike TF-IDF, which captures word frequency and surface-level patterns, GPT embeddings encode **semantic meaning and contextual relationships** between words. As a result, articles with similar meanings are expected to be closer in the embedding space, even if they use different vocabulary.

The embeddings (`X_embed`) are high-dimensional (1536 dimensions), and are normalized before clustering to ensure that similarity is based on direction rather than magnitude.

This allows us to investigate whether semantic representations produce clearer clustering structure compared to TF-IDF-based features.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt

# Evaluate different values of k for K-Means
for k in [5, 10, 15]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_embed_norm)

    # Silhouette score measures how well-separated the clusters are
    # Higher values indicate better separation; low values suggest overlap
    score = silhouette_score(X_embed_norm, labels, metric="cosine")
    print(f"k = {k}, silhouette_score = {score:.4f}")

# k = 15 gives the best silhouette score, but the score is still low.
# This suggests that the embedding space contains overlapping semantic structure
# rather than sharply separated clusters.

kmeans_embed = KMeans(n_clusters=10, random_state=42, n_init=10)
labels_embed = kmeans_embed.fit_predict(X_embed_norm)
print(pd.Series(labels_embed).value_counts())
df_sample["embed_cluster"] = labels_embed

# Compute the average fake-news proportion in each cluster
print(df_sample.groupby("embed_cluster")["label"].mean().sort_values(ascending=False))

# PCA projection for visualization
pca = PCA(n_components=2, random_state=42)
X_pca2 = pca.fit_transform(X_embed_norm)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=labels_embed, cmap="tab10", s=5)
plt.title("Embedding K-Means Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# UMAP projection for visualization
reducer = umap.UMAP(random_state=42)
X_umap2 = reducer.fit_transform(X_embed_norm)
plt.figure(figsize=(8, 6))
plt.scatter(X_umap2[:, 0], X_umap2[:, 1], c=labels_embed, cmap="tab10", s=5)
plt.title("Embedding K-Means Clusters (UMAP)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.show()

## HDBSCAN Clustering

HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise) is a density-based clustering algorithm that identifies clusters based on regions of high data density.

Unlike K-Means, HDBSCAN does not require specifying the number of clusters in advance. Instead, it automatically detects clusters of varying shapes and sizes, while labeling low-density points as **noise**.

### How HDBSCAN Works

The algorithm follows these key steps:

1. Estimate the **local density** of each data point
2. Build a hierarchy of clusters based on density connectivity
3. Extract the most stable clusters from the hierarchy
4. Label points that do not belong to any dense region as **noise**

### Objective Function

HDBSCAN does not optimize a single global objective like K-Means. Instead, it focuses on:

- identifying **dense regions** in the data
- separating them from **sparse regions (noise)**
- selecting clusters that are stable across different density levels

### Key Characteristics

- Produces **soft clustering with noise detection** (points can be labeled as -1 if they do not belong to any cluster)
- Does **not require pre-specifying the number of clusters**
- Can detect:
  - clusters of varying shapes
  - clusters of different densities

### Limitations

- Sensitive to hyperparameters:
  - `min_cluster_size`
  - `min_samples`
- May label a large portion of data as noise if clusters are not well-defined
- Can struggle in very high-dimensional spaces

### Application in This Project

In this project, HDBSCAN is used to evaluate whether the embedding space contains **well-defined dense clusters** of news articles.

If meaningful clusters exist, HDBSCAN should identify them. However, if most points are labeled as noise, this suggests that the data is not organized into clearly separated density regions, but instead forms **continuous semantic gradients**.

In [ ]:
import hdbscan

# Test different HDBSCAN settings
# min_cluster_size = smallest group size required to form a cluster
# min_samples = how conservative the algorithm is when deciding whether a point belongs to a dense region
for mcs in [30, 50, 100]:
    for ms in [5, 10, 20]:
        hdb = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms
        )
        labels = hdb.fit_predict(X_embed_norm)

        # Count the number of clusters, excluding noise label (-1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        # Proportion of points labeled as noise
        noise_ratio = (labels == -1).mean()
        print(f"min_cluster_size={mcs}, min_samples={ms}, clusters={n_clusters}, noise_ratio={noise_ratio:.3f}")

# Results show that HDBSCAN finds very few clusters and labels most points as noise.
# This suggests that the embedding space does not contain sharp density-based clusters,
# but instead has overlapping and continuous semantic structure.

### Interpreting Cluster Narratives with GPT

To better understand the semantic structure of each cluster, I use a large language model (GPT) to analyze a sample of articles from the cluster.

While clustering and word frequency analysis reveal statistical patterns, they do not directly explain the **underlying narrative or rhetorical style** of the content.

By providing representative excerpts to a language model, we can generate a higher-level interpretation of:
- the dominant narrative themes
- the tone and rhetorical framing
- potential biases or patterns in the content

This step complements quantitative analysis with qualitative interpretation.

In [ ]:
# Sample texts from a cluster
sample_texts = cluster_7_docs["text"].sample(10, random_state=42).tolist()

# Prompt for GPT
prompt = """
Below are excerpts from a cluster of news articles.
Please summarize:
1. The underlying narrative theme
2. The rhetorical style (e.g., neutral, emotional, persuasive, conspiratorial)
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You analyze news narratives and rhetorical patterns."},
        {"role": "user", "content": prompt + "\n\n".join(sample_texts)}
    ]
)

print(response.choices[0].message.content)

### Interpretation
The GPT-generated summary suggests that this cluster is characterized by a consistent narrative theme and rhetorical style.
This indicates that clustering is not only grouping articles based on vocabulary, but also capturing deeper narrative patterns. These patterns may help explain why supervised models perform well, as they can leverage recurring themes and framing strategies present in the data.

### Note
The GPT-based interpretation is used as a qualitative tool to aid understanding of cluster structure. It should not be considered ground truth, but rather a helpful summary of patterns present in the sampled texts.

## Gaussian Mixture Model (GMM)

Gaussian Mixture Model (GMM) is a probabilistic clustering algorithm that assumes the data is generated from a mixture of multiple Gaussian distributions.

Unlike K-Means, which assigns each data point to a single cluster, GMM provides **soft cluster assignments**, meaning each data point has a probability of belonging to each cluster.

### How GMM Works

The algorithm uses an iterative process called the Expectation-Maximization (EM) algorithm:

1. **Initialize parameters** (means, covariances, and mixture weights)
2. **Expectation step (E-step)**:
   - Compute the probability that each data point belongs to each cluster
3. **Maximization step (M-step)**:
   - Update cluster parameters based on these probabilities
4. Repeat until convergence

### Objective Function

GMM aims to maximize the **likelihood of the data** under the mixture model:

- Each cluster is modeled as a Gaussian distribution
- The model finds parameters that best explain the observed data

### Key Characteristics

- Produces **soft probabilistic assignments**
- Can model:
  - overlapping clusters
  - elliptical cluster shapes (depending on covariance type)
- More flexible than K-Means

### Limitations

- Requires specifying the number of components **K**
- Sensitive to initialization
- Can be computationally more expensive than K-Means
- Assumes data follows Gaussian-like distributions

### Application in This Project

In this project, GMM is used to model the **overlapping structure** of fake and true news in embedding space.

Instead of forcing each article into a single cluster, GMM assigns a probability distribution over clusters. These probabilities are then used to construct a **fake risk score**, which reflects how strongly an article is associated with clusters dominated by fake news.

This approach better captures the **continuous and uncertain nature of misinformation**, where articles may not belong entirely to a single category.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --------------------------------------------------
# Evaluate different values of K for GMM
# --------------------------------------------------

# Use normalized embeddings for clustering

for k in [5, 8, 10, 12, 15]:
    gmm = GaussianMixture(
        n_components=k,
        covariance_type="diag",   # diagonal covariance for high-dimensional embeddings
        random_state=42
    )
    labels = gmm.fit_predict(X_embed_norm)

    print(f"\n=== k = {k} ===")

    # 1) Cluster sizes
    print("Cluster counts:")
    print(pd.Series(labels).value_counts())

    # 2) Fake proportion per cluster
    df_temp = df_sub.copy()
    df_temp["gmm_cluster"] = labels
    fake_props = (
        df_temp.groupby("gmm_cluster")["label"]
        .mean()
        .sort_values(ascending=False)
    )

    print("\nFake proportion per cluster:")
    print(fake_props)

    # 3) Coverage of fake articles in high-purity clusters
    # High-purity clusters are clusters where the mean fake label is > 0.8
    mask = df_temp.groupby("gmm_cluster")["label"].transform("mean") > 0.8
    coverage = df_temp[mask]["label"].sum() / df_temp["label"].sum()

    print(f"\nFake coverage (>80% fake clusters): {coverage:.3f}")
    print("-" * 50)

# --------------------------------------------------
# Fit final GMM with k = 10
# --------------------------------------------------

# I selected k = 10 because it provided a stable set of clusters while maintaining reasonable fake-news coverage in high-purity clusters. This makes it a better choice for constructing a cluster-based risk score than larger values of k, which produced smaller but purer clusters with lower coverage.
gmm = GaussianMixture(
    n_components=10,
    covariance_type="diag",
    random_state=42
)

labels_gmm = gmm.fit_predict(X_embed_norm)
probs_gmm = gmm.predict_proba(X_embed_norm)   # shape: (6000, 10)

# Compute fake rate for each cluster
df_sub = df_sub.copy()
df_sub["gmm_cluster"] = labels_gmm

cluster_fake_rate = (
    df_sub.groupby("gmm_cluster")["label"]
    .mean()                                  # average fake proportion in each cluster
    .reindex(range(gmm.n_components))        # keep cluster order consistent
    .values
)

print("Cluster fake rates:")
print(cluster_fake_rate)

# Example interpretation:
# If an article belongs to cluster 9, that cluster contains a high proportion of fake articles.

# --------------------------------------------------
# Convert soft cluster membership into a continuous risk score
# --------------------------------------------------
# Each article receives a weighted fake-risk score based on its
# membership probabilities across all clusters.
risk_score = probs_gmm @ cluster_fake_rate
df_sub["fake_risk_score"] = risk_score

print(df_sub.groupby("label")["fake_risk_score"].mean())
# label = 0 (true) should have a lower average risk score
# label = 1 (fake) should have a higher average risk score

### Comparison of Clustering Methods

- **K-Means** reveals stable partitions but assumes simple cluster shapes
- **HDBSCAN** tests for density-based structure and identifies noise
- **GMM** captures overlapping and probabilistic structure

Together, these methods provide complementary views of the data, highlighting that fake and true news are not perfectly separable, but instead exist along a continuous semantic spectrum.